# Parallel Sets (or Parallel Categories)
Parallel Sets is a visualization technique that plots data into rectangular, typically horizontal boxes. The size of the boxes represents the frequency or proportion of the entities in the category.

### Properties
- technique to visualize multivariate categorical data by color and proportion
Method
- rectangular boxes replace axes in parallel coordinates
(relative) size represents the frequency of data
- connect categories with lines to visualize dependencies

We visualize the Titanic data set using parallel sets as implemented in the Python visualization package **Plotly** (Plotly Open Source Graphing Library for Python)

In [2]:
import plotly.express as px
import pandas as pd
import numpy as np

import nbformat as nbf
print(nbf.__version__)


titanic = pd.read_csv('../../data/titanic.csv')
# don't like missing values, set as unknown 
titanic.fillna(-1, inplace=True)

# modify the data frame to add a new column for Age Category and Class
titanic.insert(
    titanic.columns.get_loc('Age'), 
    'Age Category', 
    pd.cut(titanic['Age'], bins=[-10, 0, 17, 100], labels=['Age unknown', 'Child', 'Adult'])
    )
titanic.insert(
    titanic.columns.get_loc('Pclass'), 
    'Class', 
    pd.cut(titanic['Pclass'], bins=[0, 1, 2, 3], labels=['1st class', '2nd class', '3rd class'])
    )

# set colors for the categories
colors = ['maroon', 'lightsteelblue', 'mediumseagreen']
# create the parallel categories plot
fig = px.parallel_categories(titanic, dimensions=['Survived', 'Class', 'Sex', 'Age Category'], 
                             color='Survived', 
                             color_continuous_scale=colors, #cale,
                             labels={'Survived': 'Survived', 'Age Category': 'Age', 'Class': 'Class', 'Sex': 'Gender'},
                             title='Titanic Passenger Data')
# do not plot color scale, it is meaningless
fig.update_coloraxes(showscale=False)
# done!
fig.show()

5.10.4


### Sample Example but using Plotly Graphic Objects

In [ ]:
# This implementation is based on the the example from the official documentation
# https://plotly.com/python/parallel-categories-diagram/
import plotly.graph_objects as go

# Create dimensions
class_dim = go.parcats.Dimension(
    values=titanic['Pclass'],
    label="Class",
    categoryarray=[1, 2, 3],
    ticktext=['1st class', '2nd class', '1rd class'],
)
gender_dim = go.parcats.Dimension(values=titanic['Sex'], label="Gender")
age_dim =  go.parcats.Dimension(values=titanic['Age Category'], label="Age")
survival_dim = go.parcats.Dimension(
    values=titanic['Survived'], label="Survived",
    categoryarray=[0, 1],
    ticktext=['perished', 'survived']
)

# Create color scale
colors = ['maroon', 'lightsteelblue', 'mediumseagreen']
# Create figure: parallel categories plot
fig = go.Figure(data = [go.Parcats(dimensions=[class_dim, age_dim, gender_dim, survival_dim],
        line={'color': titanic['Survived'], 'colorscale': colors, 'shape': 'hspline'},
        hoveron='color', hoverinfo='count+probability',
        labelfont={'size': 18, 'family': 'Times'},
        tickfont={'size': 16, 'family': 'Times'},
        arrangement='freeform')]
)

fig.show()

# Multiple Linked Views with Brushing

In [ ]:
#import anywidget
#from ipywidgets import widgets
import plotly.io as pio
#pio.renderers.default = 'vscode'
#pio.renderers.default = 'notebook_connected'
# pio.rendersers.default = 'jupyterlab'
#pio.renderers.default = 'jupyterlab'

# try first the car import dataset 
cars_df = pd.read_csv('../../data/imports-85.csv')

# set categories for the columns
categories = ['body-style', 'drive-wheels', 'engine-location', 'num-of-cylinders', 'fuel-type']

# create a list of dimensions
dimensions = [dict(values=cars_df[cat], label=cat) for cat in categories]

# colors
color = np.zeros(len(cars_df), dtype='uint8') 
colors = ['maroon', 'lightsteelblue', 'mediumseagreen']

# Figure: use widgets to create multiple linked views
fig = go.FigureWidget(
    data = [
        go.Scatter(x=cars_df['horsepower'], y=cars_df['highway-mpg'], 
                   mode='markers', 
                   marker={'color':'cadetblue', 'size': 10, 'opacity': 0.4},
                   selected={'marker': {'color': 'firebrick', 'size': 20, 'opacity': 0.8}},
                   unselected={'marker': {'color': 'cadetblue', 'size': 10, 'opacity': 0.4}}
        ),
        go.Parcats(
            domain={'y': [0, 0.4]}, dimensions=dimensions,
        line={'colorscale': colors, 'cmin': 0,
              'cmax': 1, 'color': color, 'shape': 'hspline'})                   
    ]
)

fig.update_layout(
        height=800, xaxis={'title': 'Horsepower'},
        yaxis={'title': 'MPG', 'domain': [0.6, 1]},
        dragmode='lasso', hovermode='closest')

# Update color callback
def update_color(trace, points, state):
   # Update scatter selection
    fig.data[0].selectedpoints = points.point_inds

    # Update parcats colors
    new_color = np.zeros(len(cars_df), dtype='uint8')
    new_color[points.point_inds] = 1
    fig.data[1].line.color = new_color

# Register callback on scatter selection...
fig.data[0].on_selection(update_color)
# and parcats click
fig.data[1].on_click(update_color)

fig 
#fig.show()
#fig.show(renderer='browser')